# Flat Maintenance Defaulters WhatsApp Notifier



This notebook identifies defaulters from a Google Sheet, looks up their contact numbers from Google Contacts, and sends WhatsApp notifications via WhatsApp Web.



## Workflow

1. Read Google Spreadsheet (flat numbers, owner names, monthly payment status)

2. Identify defaulters (owners with unpaid months)

3. Look up phone numbers from Google Contacts using People API

4. Send WhatsApp messages via WhatsApp Web with payment reminders

5. Export results to CSV



## Prerequisites

- Google Sheet with payment data (first 2 columns: flat number, owner name; remaining columns: months April–March)

- Google Account with Sheets API and People API enabled

- WhatsApp Account (WhatsApp Web will be used via browser automation)

- Access to Google Contacts

## Step 1: Install Dependencies and Authenticate with Google

In [ ]:
# Install required libraries (run this cell first in Colab)

!pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib pywhatkit pandas


In [ ]:
import os

import re

from getpass import getpass

from typing import List, Dict, Optional

import pandas as pd

import time



from google.colab import auth

import google.auth

from googleapiclient.discovery import build

from googleapiclient.errors import HttpError



import pywhatkit as pwk



print("Libraries imported successfully!")


In [ ]:
# Authenticate with Google (a popup will appear in Colab)
SCOPES = [
    'https://www.googleapis.com/auth/spreadsheets.readonly',
    'https://www.googleapis.com/auth/contacts.readonly'
]

print('Authenticating with Google...')
auth.authenticate_user()
creds, _ = google.auth.default(scopes=SCOPES)
print('Google authentication successful!')

## Step 2: Define Helper Functions

In [ ]:
# Helper functions: sheet reading, contacts lookup, phone normalization, WhatsApp send



def get_defaulters_from_sheet(spreadsheet_id: str, range_name: str, creds) -> pd.DataFrame:

    """Read sheet and return DataFrame of defaulters with counts of months unpaid.

    Assumptions:

      - First two columns: flat number and owner name.

      - Remaining columns are months (April..March). Empty or zero-like values = unpaid.

    """

    sheets = build('sheets', 'v4', credentials=creds)

    try:

        result = sheets.spreadsheets().values().get(spreadsheetId=spreadsheet_id, range=range_name).execute()

    except HttpError as e:

        raise RuntimeError(f"Sheets API error: {e}")



    values = result.get('values', [])

    if not values:

        raise RuntimeError('No data found in sheet')



    header = values[0]

    rows = values[1:]



    month_cols = list(range(2, max(2, len(header))))

    month_names = header[2:]



    data = []

    for r in rows:

        flat = r[0] if len(r) > 0 else ''

        owner = r[1] if len(r) > 1 else ''

        unpaid = 0

        for ci in month_cols:

            if ci >= len(r):

                unpaid += 1

            else:

                v = str(r[ci]).strip()

                if v == '' or re.fullmatch(r'0+(?:\.0+)?', v):

                    unpaid += 1

        data.append({'flat': flat, 'owner': owner, 'months_not_paid': unpaid})



    df = pd.DataFrame(data)

    df = df[df['months_not_paid'] > 0].reset_index(drop=True)

    df.attrs['month_names'] = month_names

    return df





def get_contacts_map(creds) -> Dict[str, List[str]]:

    people = build('people', 'v1', credentials=creds)

    contacts = {}

    page_token = None

    while True:

        try:

            resp = people.people().connections().list(

                resourceName='people/me',

                personFields='names,phoneNumbers',

                pageSize=1000,

                pageToken=page_token

            ).execute()

        except HttpError as e:

            raise RuntimeError(f"People API error: {e}")



        connections = resp.get('connections', [])

        for p in connections:

            names = p.get('names', [])

            phones = p.get('phoneNumbers', [])

            if not names or not phones:

                continue

            display = names[0].get('displayName', '').strip().lower()

            phone_vals = [ph.get('value', '').strip() for ph in phones if ph.get('value')]

            if phone_vals:

                contacts.setdefault(display, []).extend(phone_vals)



        page_token = resp.get('nextPageToken')

        if not page_token:

            break

    return contacts





def find_phone_for_name(name: str, contacts_map: Dict[str, List[str]]) -> Optional[List[str]]:

    if not name:

        return None

    name_l = name.strip().lower()

    if name_l in contacts_map:

        return contacts_map[name_l]

    for k, v in contacts_map.items():

        if name_l in k or k in name_l:

            return v

    tokens = [t for t in re.split(r"\s+", name_l) if t]

    for t in tokens:

        for k, v in contacts_map.items():

            if t in k.split():

                return v

    return None





def normalize_phone(phone: str) -> str:

    p = re.sub(r"[^0-9+]", "", phone)

    if p.startswith('00'):

        p = '+' + p[2:]

    if p.startswith('+'):

        return p

    return p





def send_whatsapp_web(to_phone: str, message: str, wait_time: int = 15):

    """Send WhatsApp message via WhatsApp Web (browser automation).

    Args:

        to_phone: Phone number with country code (e.g., '+919876543210')

        message: Message text to send

        wait_time: Time to wait before closing browser (default 15 seconds)

    Returns:

        Boolean indicating success

    """

    try:

        # Remove 'whatsapp:' prefix if present

        phone = to_phone.replace('whatsapp:', '')

        # pywhatkit requires 15 second minimum wait

        pwk.sendwhatmsg_instantly(phone, message, wait_time=wait_time, tab_close=True)

        return True

    except Exception as e:

        print(f"Error sending WhatsApp to {to_phone}: {e}")

        return False


## Step 3: Read Spreadsheet and Identify Defaulters

In [ ]:
spreadsheet_id = input('Enter Spreadsheet ID: ').strip()

if not spreadsheet_id:

    raise SystemExit('Spreadsheet ID required')



range_name = input('Enter sheet range (default "Sheet1!A1:Z"): ').strip() or 'Sheet1!A1:Z'



print('Reading sheet and computing defaulters...')

df_def = get_defaulters_from_sheet(spreadsheet_id, range_name, creds)

if df_def.empty:

    print('No defaulters found. Exiting.')

else:

    print(f'Found {len(df_def)} defaulter(s):')

    print(df_def)

## Step 4: Look Up Contact Numbers from Google Contacts

In [ ]:
print('Fetching contacts from Google Contacts...')

contacts_map = get_contacts_map(creds)

print(f'Fetched {len(contacts_map)} unique contacts from Google Contacts')



# Match defaulters with contacts

df_def['phone_numbers'] = df_def['owner'].apply(lambda name: find_phone_for_name(name, contacts_map))

df_def['phone_normalized'] = df_def['phone_numbers'].apply(lambda phones: [normalize_phone(p) for p in phones] if phones else [])

df_def['phone_found'] = df_def['phone_normalized'].apply(lambda phones: len(phones) > 0)



print('\nDefaulters with phone numbers found:')

print(df_def[df_def['phone_found']])



print('\nDefaulters WITHOUT phone numbers:')

print(df_def[~df_def['phone_found']])

## Step 5: Send WhatsApp Messages via WhatsApp Web (Optional)

In [ ]:
# WhatsApp Web setup

use_whatsapp_web = input('Do you want to send WhatsApp messages via WhatsApp Web now? (y/N): ').strip().lower() == 'y'



if use_whatsapp_web:

    print('\u26a0️  NOTE: WhatsApp Web will open in a browser. You must be logged in to send messages.')

    print('Make sure your WhatsApp Web is accessible before proceeding.\n')

    

    template = input('Message template (use {name} and {months} placeholders):\n')

    if not template:

        template = 'Dear {name}, our records show {months} month(s) unpaid. Please clear your dues.'

    

    results = []

    contact_count = 0

    

    for _, row in df_def[df_def['phone_found']].iterrows():

        owner = row['owner']

        flat = row['flat']

        months = int(row['months_not_paid'])

        phones = row['phone_normalized']

        msg_text = template.format(name=owner or flat, months=months)

        

        sent = False

        if phones:

            contact_count += 1

            print(f"\nSending message to {owner} ({phones[0]})...")

            sent = send_whatsapp_web(phones[0], msg_text)

            if sent:

                print(f'✓ Message sent to {owner}')

            else:

                print(f'✗ Failed to send to {owner}')

        

        results.append({'flat': flat, 'owner': owner, 'months_not_paid': months, 'phone': phones[0] if phones else None, 'sent': sent})

    

    res_df = pd.DataFrame(results)

    print(f'\n=== SUMMARY ===' )

    print(res_df)

    print(f'\nTotal messages sent: {res_df["sent"].sum()} / {len(res_df)}')

else:

    print('Skipping WhatsApp messages. You can export the contact list and send messages manually.')


## Step 6: Export Results to CSV

In [ ]:
# Export full defaulters list with phone numbers to CSV

export_df = df_def[['flat', 'owner', 'months_not_paid', 'phone_normalized', 'phone_found']].copy()

export_df['phone_normalized'] = export_df['phone_normalized'].apply(lambda x: ', '.join(x) if x else '')



csv_file = 'defaulters_report.csv'

export_df.to_csv(csv_file, index=False)

print(f'✓ Full defaulters report exported to {csv_file}')



# If messages were sent, also save the results

if use_whatsapp_web and 'res_df' in locals():

    msg_csv_file = 'defaulters_whatsapp_sent.csv'

    res_df.to_csv(msg_csv_file, index=False)

    print(f'✓ Message send results exported to {msg_csv_file}')



print('\nDone! Download the CSV files from the Files pane in Colab.')
